In [0]:
import os
import shutil
import pandas as pd
import pyarrow as pa

# show all columns when printing DataFrames
pd.set_option("display.max_columns", None)

In [0]:
# Create widgets with default values
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("silver_schema", "silver", "Schema")
dbutils.widgets.text("bronze_schema", "bronze", "Schema")

# Then retrieve the values
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")
spark.sql(f"USE {catalog}.{silver_schema}")

In [0]:


SCHEMA = pa.schema([
    ("flight_number", pa.string()),
    ("airline", pa.string()),
    ("origin", pa.string()),
    ("destination", pa.string()),
    ("search_route", pa.string()),
    ("departure_time", pa.string()),
    ("arrival_time", pa.string()),
    ("duration_minutes", pa.int64()),
    ("stops", pa.int64()),
    ("price", pa.float64()),
    ("currency", pa.string()),
    ("trip_type", pa.string()),
    ("fetched_at", pa.string()),
])

In [0]:
MONITORING_CATALOG = "monitoring"
MONITORING_SCHEMA = "pipeline_ops"

In [0]:
def read_bronze(table_name):
    # Read a Bronze Delta table into a DataFrame
    try:
        df = spark.read.table(table_name)
        pdf = df.toPandas()
        print(f"[silver] {table_name} loaded: {len(pdf)} rows")
        return pdf
    except Exception as e:
        print(f"[silver] {table_name} not found — run Bronze script first")
        return pd.DataFrame()

In [0]:
def clean(df):
    # Add search_route column and convert duration string to total minutes
    df["search_route"] = df["origin"] + " → " + df["destination"]
    df["duration_minutes"] = pd.to_numeric(df["duration"], errors="coerce").fillna(0).astype(int)
    df = df.drop(columns=["duration"])
    df = df.drop_duplicates(subset=["flight_number", "departure_time", "trip_type"])
    df = df[[
        "flight_number", "airline", "origin", "destination", "search_route",
        "departure_time", "arrival_time", "duration_minutes", "stops",
        "price", "currency", "trip_type", "fetched_at"
    ]]
    print(f"[silver] cleaned rows: {len(df)}")
    return df

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType, LongType, BooleanType
from datetime import datetime, timezone

def write_pipeline_metrics(pipeline_name, started_at, rows_processed,
    rows_quarantined, run_id, pipeline_version='1.0',
    slo_target_minutes=45, status='success', error_message=None
):
    # Import the catalog/schema variables from Cell 3
    from __main__ import MONITORING_CATALOG, MONITORING_SCHEMA
    
    # Define the metrics table name inside the function
    METRICS_TABLE = f"{MONITORING_CATALOG}.{MONITORING_SCHEMA}.etl_metrics"
    
    completed_at = datetime.now(timezone.utc)  # Changed from datetime.utcnow()
    duration_minutes = (completed_at - started_at).total_seconds() / 60
    slo_met = duration_minutes <= slo_target_minutes

    # Define the schema explicitly
    schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("pipeline_name", StringType(), False),
        StructField("pipeline_version", StringType(), False),
        StructField("started_at", TimestampType(), False),
        StructField("completed_at", TimestampType(), False),
        StructField("duration_minutes", DoubleType(), False),
        StructField("rows_processed", LongType(), False),
        StructField("rows_quarantined", LongType(), False),
        StructField("slo_target_minutes", IntegerType(), False),
        StructField("slo_met", BooleanType(), False),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)  # Nullable
    ])

    row = Row(
        run_id=run_id,
        pipeline_name=pipeline_name,
        pipeline_version=pipeline_version,
        started_at=started_at,
        completed_at=completed_at,
        duration_minutes=round(duration_minutes, 2),
        rows_processed=rows_processed,
        rows_quarantined=rows_quarantined,
        slo_target_minutes=slo_target_minutes,
        slo_met=slo_met,
        status=status,
        error_message=error_message
    )

    # Create DataFrame with explicit schema
    (spark.createDataFrame([row], schema=schema)
        .write.format('delta')
        .mode('append')
        .saveAsTable(METRICS_TABLE))
    
    return duration_minutes, slo_met

In [0]:


import uuid
from datetime import datetime, timezone

def write_silver(df, table_name):
    # Write cleaned DataFrame to Unity Catalog table
    try:
        # Convert pandas DataFrame to Spark DataFrame
        spark_df = spark.createDataFrame(df)
        
        # Write to Unity Catalog using Spark's native Delta support
        spark_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)
        
        print(f"[silver] {len(df)} rows written to {table_name}")
    except Exception as e:
        print(f"[silver] failed to write {table_name} to Unity Catalog")
        print(e)


# Use correct table names from Cell 4
outbound = read_bronze(f"{catalog}.{bronze_schema}.airflights_outbound")
returns = read_bronze(f"{catalog}.{bronze_schema}.airflights_return")

combined = pd.concat([outbound, returns], ignore_index=True)
cleaned = clean(combined)

direct = cleaned[cleaned["stops"] == 0]
connecting = cleaned[cleaned["stops"] > 0]

print(f"[silver] direct flights: {len(direct)} | connecting flights: {len(connecting)}")

write_silver(direct, f"{catalog}.{silver_schema}.airflights_direct")
write_silver(connecting, f"{catalog}.{silver_schema}.airflights_connecting")

start_time = datetime.now(timezone.utc)
run_id = str(uuid.uuid4())  # Generate unique run ID

# Write pipeline metrics as the last step
write_pipeline_metrics(
    pipeline_name="AirFlight_Silver",
    pipeline_version="1.0",
    started_at=start_time,
    rows_processed=len(combined),
    rows_quarantined=0,
    run_id=run_id
)